In [3]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

matplotlib version: 3.10.0
torch version: 2.5.1
tiktoken version: 0.9.0


In [26]:
import torch
from torch import nn
import torch.nn.functional as F
import tiktoken

In [5]:
torch.set_printoptions(sci_mode=False)
torch.manual_seed(123)

In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# Layer Normalization

In [7]:
class LayereNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / (std + self.eps)
        return self.scale * norm_x + self.shift

In [8]:
# test LayerNorm
test = LayereNorm(10)
x = torch.randn(2, 10)
(test(x)).mean(), (test(x)).std()


(tensor(0., grad_fn=<MeanBackward0>), tensor(1.0260, grad_fn=<StdBackward0>))

# Feed Forward

In [9]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], cfg["emb_dim"] * 4),
            nn.GELU(),
            nn.Linear(cfg["emb_dim"] * 4, cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [10]:
# test feed forward
ff = FeedForward(GPT_CONFIG_124M)
x = torch.randn(2, 3, 768)
ff(x).shape, ff(x).mean(), ff(x).std()

(torch.Size([2, 3, 768]),
 tensor(0.0010, grad_fn=<MeanBackward0>),
 tensor(0.1988, grad_fn=<StdBackward0>))

# Multi-head Attention

In [11]:
# from previous chapter
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length=6, dropout=0.0, num_heads=2, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.out_proj = nn.Linear(d_out, d_out) # d_out*d_out + d_out(bias) trainable parameters
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        
        attention_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = F.softmax(attention_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

# Transformer

In [12]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"], cfg["drop_rate"])
        self.norm1 = LayereNorm(cfg["emb_dim"])
        self.ff = FeedForward(cfg)
        self.norm2 = LayereNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        
    def forward(self, x):
        # shortcut for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        
        # shortcut for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x
        

In [13]:
x = torch.randn(2, 3, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
print("Transformer block input shape:", x.shape)
print("Transformer block output shape:", output.shape)

Transformer block input shape: torch.Size([2, 3, 768])
Transformer block output shape: torch.Size([2, 3, 768])


# GPT model block

In [14]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emd = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emd = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emd = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayereNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emd(in_idx)
        pos_emb = self.pos_emd(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop_emd(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logis = self.out_head(x)
        return logis

In [15]:
# test GPT model
model = GPTModel(GPT_CONFIG_124M)
batch = torch.randint(0, 50257, (2, 768))
#model.cfg = GPT_CONFIG_124M 
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[20228, 38554, 35778,  ...,  1110, 11237, 10059],
        [41756,  2447, 37112,  ..., 40467, 16153, 36292]])

Output shape: torch.Size([2, 768, 50257])
tensor([[[ 0.7685, -0.2585, -0.0358,  ...,  0.4785,  0.1489, -0.2942],
         [-0.2065,  0.2302,  0.1889,  ...,  0.5142,  0.0709,  0.3831],
         [-1.3560, -0.5799,  0.4483,  ...,  0.5118,  0.5837, -0.9824],
         ...,
         [ 0.2466, -1.1473, -0.0590,  ...,  0.5736,  0.7795, -0.2182],
         [-0.4430,  0.4375,  0.7270,  ..., -0.5873, -0.1437, -0.1864],
         [ 0.3697,  0.3230,  0.3667,  ..., -0.1913,  0.3059, -0.0440]],

        [[-0.0533,  0.2544,  1.0418,  ...,  0.7849,  0.7711,  0.0900],
         [-0.0979, -0.6786,  0.5675,  ...,  0.7416,  0.5374,  0.2123],
         [-0.7584, -0.0906, -0.0613,  ...,  0.2744,  0.5019, -1.2492],
         ...,
         [ 0.3022, -0.1016,  0.0440,  ...,  0.0917,  0.2400, -0.2391],
         [ 0.0437, -0.7426,  0.0176,  ...,  0.3186,  0.6988, -0.5799],
         [ 1.32

In [16]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 163,009,536


In [20]:
total_size_bytes = total_params * 4
total_size_mb = total_size_bytes / 1024**2
print(f"Total size in bytes: {total_size_mb:.2f}MB")

Total size in bytes: 621.83MB


# Generate Text

In [27]:
tokenizer = tiktoken.get_encoding("gpt2")

In [33]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx
        

In [34]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [35]:
model.eval() # disable dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=23, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716,  9646, 24274, 42322, 28832, 43581,  4619,
          6164, 13982, 47787, 27641, 37212, 13031, 43225,  5825, 19382, 23001,
          3565, 16064, 10919,  6315,  4331, 11708,  4495]])
Output length: 27


In [36]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am trackingInternationalPersonally Awakening Canterbury Since minesized snapshots5000 CicMicro Leading Temvoidchoesares Germanswhativery predictGoogle conference
